# AF-CALVADOS

This colab runs single-chain simulations with AF-CALVADOS and performs basic analysis of the resulting trajectory. The data and figures can be downloaded at the bottom of the notebook.

If you use AF-CALVADOS, please cite:

S. von Buelow, K. E. Johansson, K. Lindorff-Larsen AF-CALVADOS: AlphaFold-guided simulations of multi-domain proteins at the proteome level bioRxiv 2025

More complex simulations can be run locally with the CALVADOS package. An example for running AF-CALVADOS within the CALVADOS package can be found [here](
https://github.com/KULL-Centre/CALVADOS/tree/main/examples/single_AF_CALVADOS).

# Import and functions

In [ ]:
# @title
!pip install openmm[cuda12]
!pip install git+https://github.com/KULL-Centre/CALVADOS.git@colab_compatible
!pip install py3Dmol

import os
import calvados as cal
from calvados.cfg import Config, Job, Components, load_ebi
from calvados import sim
import subprocess
import numpy as np
import numba as nb
import math
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import MDAnalysis as mda
from MDAnalysis.coordinates.PDB import PDBWriter
from MDAnalysis.analysis import align

from io import StringIO
import json
import requests
from pathlib import Path

import py3Dmol

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams['font.size'] = 6
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['ytick.minor.size'] = 1
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['xtick.minor.size'] = 1
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.dpi'] = 300
plt.rcParams['lines.markersize'] = 3
plt.rcParams['lines.markeredgewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 1.
plt.rcParams['figure.max_open_warning'] = 30

col1x = 3.425 # column
col2x = 7. # 2x column

github_folder = 'https://raw.githubusercontent.com/KULL-Centre/CALVADOS/colab_compatible'

cwd = os.getcwd()

def load_pae(name, pdb_folder):
    input_pae = f'{pdb_folder}/{name}.json'
    pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=0) / 10. # nm
    return pae

def calc_rgs(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms
    rg = cal.analysis.calc_rg(u, ag, start=start, stop=stop, step=step)
    return rg

def calc_ees(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms
    ees, _, _ = cal.analysis.calc_ete(u,ag,start=start,stop=stop,step=step)
    return ees

def calc_timesteps(name, path, start=10, stop=None, step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    dt = u.trajectory[0].dt * step
    xs = np.arange(len(u.trajectory[start:stop:step])) * dt / 1000.
    return xs

def calc_dmap_std_manual(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms

    box = u.dimensions[:3] / 10.
    coordinates = u.trajectory.timeseries(ag) / 10.

    nres = len(ag)

    dmap_std_manual = calc_dmap_std_numba(coordinates,box,nres,start=start,stop=stop,step=step)
    return dmap_std_manual

@staticmethod
@nb.jit(nopython=True)
def calc_dmap_std_numba(coordinates,box,nres,start=10,stop=None,step=1):
    dmaps_std_manual = np.zeros((nres,nres))

    for idx in range(nres):
        for jdx in range(idx,nres):
            x0 = coordinates[idx,start:stop:step]
            x1 = coordinates[jdx,start:stop:step]
            dvec = x1-x0
            dvec_pbc = np.where(dvec<=box/2, dvec, box-dvec)
            d = np.zeros((len(x0)))
            for kdx, dv_pbc in enumerate(dvec_pbc):
                d[kdx] = math.sqrt(dv_pbc[0]**2 + dv_pbc[1]**2 + dv_pbc[2]**2)
            d_std = np.std(d)

            dmaps_std_manual[idx,jdx] = d_std
            dmaps_std_manual[jdx,idx] = d_std
    return dmaps_std_manual

def load_plddt(name, pdb_folder):
    # pLDDT
    pdb = f'{pdb_folder}/{name}.pdb'
    plddt = cal.build.bfac_from_pdb(pdb)
    # bfac_matrix = np.minimum.outer(bfac,bfac)
    return plddt

################

def plot_rg(name, path, start=10, stop=None, step=1):
    rgs = calc_rgs(name, path, start=start,stop=stop,step=step)
    fig, ax = plt.subplots(1,2,figsize=(5,2))

    xs = calc_timesteps(name, path, start=10)
    ax[0].plot(xs, rgs)
    ax[0].set(xlim=(xs[0],xs[-1]))
    ax[0].set(xlabel='Time [ns]', ylabel=r'$R_\mathrm{g}$ [nm]')

    ax[1].hist(rgs,bins=50,histtype='step',color='C0',density=True)
    ax[1].hist(rgs,bins=50,alpha=0.3,color='C0',density=True)
    ax[1].set(xlabel=r'$R_\mathrm{g}$ [nm]', ylabel='pdf')

    fig.tight_layout()
    os.makedirs(f'{name}/figures',exist_ok=True)
    fig.savefig(f'{name}/figures/{name}_rg.pdf')

def plot_ee(name, path, start=10, stop=None, step=1):
    # rgs = calc_rgs(name, path, start=10,stop=None,step=1)
    ees = calc_ees(name, path, start=start,stop=stop,step=step)
    fig, ax = plt.subplots(1,2,figsize=(5,2))

    xs = calc_timesteps(name, path, start=10)
    ax[0].plot(xs, ees)
    ax[0].set(xlim=(xs[0],xs[-1]))
    ax[0].set(xlabel='Time [ns]', ylabel=r'$R_\mathrm{ee}$ [nm]')

    ax[1].hist(ees,bins=50,histtype='step',color='C0',density=True)
    ax[1].hist(ees,bins=50,alpha=0.3,color='C0',density=True)
    ax[1].set(xlabel=r'$R_\mathrm{ee}$ [nm]', ylabel='pdf')

    fig.tight_layout()
    os.makedirs(f'{name}/figures',exist_ok=True)
    fig.savefig(f'{name}/figures/{name}_ree.pdf')

def plot_plddt(name, pdb_folder):
    seq = cal.sequence.seq_from_pdb(f'{pdb_folder}/{name}.pdb')[0]
    print(seq)
    nres = len(seq)
    xs = np.arange(1,nres+1)

    plddt = load_plddt(name, pdb_folder)
    fig, ax = plt.subplots(figsize=(3,1.5))
    ax.plot(xs, plddt*100)
    ax.set(xlim=(xs[0],xs[-1]))
    ax.set(xlabel='Residue', ylabel='pLDDT')
    fig.tight_layout()
    os.makedirs(f'{name}/figures',exist_ok=True)
    fig.savefig(f'{name}/figures/{name}_pLDDT.pdf')

def plot_pae_sigma(name, path, pdb_folder, start=10, stop=None, step=1):
    pae = load_pae(name, pdb_folder)
    nres = len(pae)
    fig, ax = plt.subplots(1,2,figsize=(4,2.2))
    axij_pae = ax[0]
    _ = axij_pae.imshow(pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)

    divider = make_axes_locatable(axij_pae)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)

    xs = np.arange(0,nres,100)

    cmap = plt.cm.Blues_r

    dmaps_std = calc_dmap_std_manual(name, path, start=start, stop=stop, step=step)

    axij_dmaps = ax[1]

    _ = axij_dmaps.imshow(dmaps_std,cmap=cmap,vmin=0,vmax=2)
    axij_dmaps.set_title(f'{name}\n'+r'$\sigma(r)$ Sim.', fontsize=6, linespacing=1.5)
    divider = make_axes_locatable(axij_dmaps)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    for a in [axij_pae, axij_dmaps]:
        a.grid(False)
        a.set(xlabel='Residue',ylabel='Residue')
    fig.tight_layout()
    os.makedirs(f'{name}/figures',exist_ok=True)
    fig.savefig(f'{name}/figures/{name}_PAE_vs_sigma.pdf')

##############

def set_up_system(name, path, temp, ionic, pH, L, N_save, N_frames, pdb_folder, fresidues):

    config = Config(
      # GENERAL
      sysname = name, # name of simulation system
      box = [L, L, L], # nm
      temp = temp, # 20 degrees Celsius
      ionic = ionic, # molar
      pH = pH, # 7.5
      topol = 'center',

      # RUNTIME SETTINGS
      wfreq = N_save, # dcd writing interval, 1 = 10 fs
      steps = N_frames*N_save, # number of simulation steps
      runtime = 0, # overwrites 'steps' keyword if > 0
      platform = 'CUDA', # or CUDA
      restart = 'checkpoint',
      frestart = 'restart.chk',
      verbose = True,
    )

    # PATH
    subprocess.run(f'mkdir -p {path}',shell=True)

    config.write(path,name='config.yaml')

    components = Components(
      # Defaults
      molecule_type = 'protein',
      nmol = 1, # number of molecules
      restraint = True, # apply restraints
      charge_termini = 'both', # charge N or C or both

      # INPUT
      fresidues = fresidues, # residue definitions
      pdb_folder = pdb_folder, # directory for pdb and PAE files

      # RESTRAINTS
      restraint_type = 'go', # harmonic or go
      use_com = True, # apply on centers of mass instead of CA
      colabfold = 0, # PAE format (EBI AF=0, Colabfold=1&2)
      k_go = 15., # Restraint force constant
    )
    components.add(name=name)
    components.write(path,name='components.yaml')

####################

ALPHAFOLD_API = "https://alphafold.ebi.ac.uk/api/prediction"

def write_entry(uniprot, entry, pdb_folder):
    out = Path(pdb_folder) / f"{uniprot}_info.json"
    with open(out, 'w') as f:
        json.dump(entry, f, indent=2)

def download_file(url, dest):
    r = requests.get(url, allow_redirects=True)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)

def load_ebi(uniprot, pdb_folder):
    folder = Path(pdb_folder)
    folder.mkdir(parents=True, exist_ok=True)

    # Fetch entry metadata
    r = requests.get(f"{ALPHAFOLD_API}/{uniprot}")
    r.raise_for_status()
    results = r.json()

    if not results:
        raise ValueError(f"No AlphaFold entry found for UniProt ID: {uniprot}")

    entry = results[0]

    # Download PDB structure
    pdb_url = entry.get("pdbUrl")
    if not pdb_url:
        raise KeyError(f"'pdbUrl' not found in API response for {uniprot}")
    download_file(pdb_url, folder / f"{uniprot}.pdb")

    # Download PAE JSON — prefer new field, fall back to deprecated one
    pae_url = entry.get("paeDocUrl")
    if not pae_url:
        raise KeyError(f"No PAE doc URL found in API response for {uniprot}")
    download_file(pae_url, folder / f"{uniprot}.json")

    # Save full entry metadata
    write_entry(uniprot, entry, pdb_folder)
    print(f"Done: {uniprot}")

##############


def align_traj(pdb, traj, start=None, stop=None, step=1, reference_frame=0):
    """RMSD-align trajectory to a reference frame and write a new DCD"""

    u = mda.Universe(pdb, traj)

    # Reference structure (usually first frame)
    ref = mda.Universe(pdb, traj)
    ref.trajectory[reference_frame]

    with mda.Writer(f"{traj[:-4]}_aligned.dcd", len(u.atoms)) as W:
        for ts in u.trajectory[start:stop:step]:
            # Align current frame to reference
            align.alignto(u, ref, select="all")
            W.write(u.atoms)

#########

def postprocess_traj(name, path, step_view = 10):
    cal.analysis.center_traj(f'{path}/top.pdb', f'{path}/{name}.dcd', start=10)
    align_traj(f'{path}/top.pdb', f'{path}/{name}_c.dcd')

    u = mda.Universe(f'{path}/top.pdb', f'{path}/{name}_c_aligned.dcd')

    nres = len(u.atoms)
    models = []
    pdb_io = StringIO()

    with mda.Writer('tmp.pdb', multiframe=True, format='pdb') as W:
        for ts in u.trajectory[::step_view]:
            W.write(u.atoms)
    return nres

def visualize(nres):
    cmap = plt.cm.viridis
    view = py3Dmol.view(width=800, height=600)
    with open('tmp.pdb', 'r') as f:
        pdb_str = f.read()
        view.addModelsAsFrames(pdb_str)

    view.setStyle({"sphere": {}})
    for idx in range(nres):
        color = np.array(cmap(idx/nres)[:3])*255
        colorstr = f'rgb({color[0]}, {color[1]}, {color[2]})'
        view.setStyle({"serial": [idx+1]}, {"sphere": {"color": colorstr}})

    view.zoomTo()
    view.animate({"loop": "forward"})
    view.show()

# Input

In [ ]:
name = 'O75764' #@param {type:"string"}
temp = 293 #@param {type:"number"}
ionic = 0.15 #@param {type:"number"}
pH = 7.5 #@param {type:"number"}

step_size_ns = 0.01 #@param {type:"number", label:"Step size [ns]"}
sim_length_ns = 10. #@param {type:"number", label:"Simulation length [ns]"}

# set the side length of the cubic box
L = 80

# set the saving interval (number of integration steps)
N_save = int(step_size_ns * 100 * 1000)
N_frames = int(sim_length_ns / step_size_ns)

print(f'wfreq: {N_save}, frames: {N_frames}')

os.makedirs(f'{cwd}/{name}/input',exist_ok=True)
os.system(f'wget -O {name}/input/residues.csv {github_folder}/examples/single_AF_CALVADOS/input/residues.csv')

pdb_folder = f'{cwd}/{name}/input'
fresidues = f'{cwd}/{name}/input/residues.csv'

path = f'{cwd}/{name}/simulation'

load_ebi(name, pdb_folder)

set_up_system(name, path, temp, ionic, pH, L, N_save, N_frames, pdb_folder, fresidues)

# Run simulation

In [ ]:
sim.run(path=path,fconfig='config.yaml',fcomponents='components.yaml')

# Visualize

In [ ]:
nres = postprocess_traj(name, path)
visualize(nres)

# Analysis

## Plot pLDDT

In [ ]:
plot_plddt(name, pdb_folder)

## Plot Rg

In [ ]:
plot_rg(name, path)

## Plot end to end distances

In [ ]:
plot_ee(name, path)

## Plot PAE vs pairwise distance fluctuations

In [ ]:
plot_pae_sigma(name, path, pdb_folder)

# Download data

In [ ]:
os.makedirs(name,exist_ok=True)

os.system(f'zip -r af_calvados_{name}.zip {name}')

from google.colab import files
files.download(f'af_calvados_{name}.zip')